In [0]:
dbutils.widgets.text("catalog","fk_1361")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F

In [0]:
df = spark.read.table(f"{catalog}.bronze.customers_bronze")

In [0]:
df.display()

In [0]:
def standardize(df):
    date_formats = ["yyyy-MM-dd", "dd/MM/yyyy"]
    date_expr = [F.try_to_date(F.col("signup_date"), fmt) for fmt in date_formats]
    
    return df.withColumns({
        "customer_id": F.trim(F.upper(F.col("customer_id"))),
        "full_name": F.trim(F.initcap(F.col("full_name"))),
        "email": F.trim(F.lower(F.col("email"))),
        "city": F.trim(F.initcap(F.col("city"))),
        "tier": F.trim(F.upper(F.col("tier"))),
        "signup_date": F.coalesce(*date_expr)
    })


In [0]:
df_standard = standardize(df)

In [0]:
df_cleaned = df_standard.dropDuplicates().dropna()
df_cleaned.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customers_silver")

In [0]:
df_qurantine = df_standard.filter(F.col("customer_id").isNull() | F.col("email").isNull() | F.col("city").isNull() | F.col("tier").isNull() | F.col("signup_date").isNull())
df_qurantine.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customers_qurantine")

In [0]:
%sql
select count(*) from fk_1361.silver.customers_silver

In [0]:
%sql
select count(*) from fk_1361.silver.customers_qurantine